# Production hierarchical hybrid modal classifier

Official model: N1 Gradient Boosting, N2 Random Forest, and N3 Extra Trees. The validated dataset contains 114 physical trips and 445 scenarios; Raw/L1/L2/L3 variants remain grouped by physical trip.

In [ ]:
import sys, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report, confusion_matrix
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 13
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11
ROOT = Path.cwd()
while not (ROOT / 'pipeline_v4').exists(): ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT/'pipeline_v4/calibration_and_diagnostics/modal_classification/calibration/hybrid')]
from pipeline_v4.src import config
from pipeline_v4.src.random_forest_contract import N1_FEATURES, N2_FEATURES, N3_FEATURES, HYBRID_HYPERPARAMETERS, BUS_PROBABILITY_THRESHOLD
from pipeline_v4.src.modal_classification import create_modal_evaluator
from experimentar_n1_caminar import make_features, M2I, MODES, WALK


## Dataset and mixed-label exclusion

In [ ]:
df = make_features('datos_entrenamiento_ml_expanded.pkl')
assert df.caid_trip.nunique() == 114 and len(df) == 445
clean = pd.read_csv(config.GPS_DIR/'Datos de MATLAB GPS Limpios.csv')
mixed = {f'{c}_{int(float(t))}' for (c,t),g in clean.groupby(['caid','num_trip']) if g.mode_of_transport.dropna().astype(str).str.strip().str.lower().nunique() > 1}
assert set(df.caid_trip).isdisjoint(mixed)
display(df.groupby('label').caid_trip.nunique().rename('viajes'))
display(df.label.value_counts().rename('escenarios'))


## Ordered feature contracts: N1=16, N2=52, N3=25

In [ ]:
print('N1', len(N1_FEATURES), list(N1_FEATURES))
print('N2', len(N2_FEATURES), list(N2_FEATURES))
print('N3', len(N3_FEATURES), list(N3_FEATURES))
print('Umbral Bus', BUS_PROBABILITY_THRESHOLD)


## Grouped validation of the official cascade

In [ ]:
X = df[list(N2_FEATURES)]; y = df.label.map(M2I).astype(int); groups = df.caid_trip
oof = np.zeros(len(df), dtype=int); folds = np.zeros(len(df), dtype=int)
cv = StratifiedGroupKFold(5, shuffle=True, random_state=42)
for fold, (tr, te) in enumerate(cv.split(X, y, groups)):
    assert set(groups.iloc[tr]).isdisjoint(set(groups.iloc[te]))
    n1 = GradientBoostingClassifier(**HYBRID_HYPERPARAMETERS['n1']).fit(X.iloc[tr][list(N1_FEATURES)], (y.iloc[tr] != WALK).astype(int))
    motor = y.iloc[tr] != WALK
    n2 = RandomForestClassifier(**HYBRID_HYPERPARAMETERS['n2']).fit(X.iloc[tr][motor][list(N2_FEATURES)], (y.iloc[tr][motor] == 2).astype(int))
    road = y.iloc[tr].isin([0, 1])
    n3 = ExtraTreesClassifier(**HYBRID_HYPERPARAMETERS['n3']).fit(X.iloc[tr][road][list(N3_FEATURES)], (y.iloc[tr][road] == 1).astype(int))
    p1 = n1.predict(X.iloc[te][list(N1_FEATURES)]); pred = np.full(len(te), WALK); mot = np.flatnonzero(p1 == 1)
    p2 = n2.predict(X.iloc[te].iloc[mot][list(N2_FEATURES)]); pred[mot[p2 == 1]] = 2; surface = mot[p2 == 0]
    pred[surface] = np.where(n3.predict_proba(X.iloc[te].iloc[surface][list(N3_FEATURES)])[:,1] >= BUS_PROBABILITY_THRESHOLD, 1, 0)
    oof[te] = pred; folds[te] = fold
print('Balanced Accuracy', balanced_accuracy_score(y, oof))
print('Macro F1', f1_score(y, oof, average='macro'))
print(classification_report(y, oof, target_names=MODES, digits=4))
cm = confusion_matrix(y, oof, labels=range(len(MODES)))
cm_normalized = confusion_matrix(y, oof, labels=range(len(MODES)), normalize="true")
display(pd.DataFrame(cm, index=MODES, columns=MODES))
display(pd.DataFrame(cm_normalized, index=MODES, columns=MODES).round(2))

def plot_confusion(values, title, filename, normalized=False):
    fig, ax = plt.subplots(figsize=(8, 6.5))
    image = ax.imshow(values, cmap="Blues", vmin=0, vmax=1 if normalized else None)
    threshold = 0.5 if normalized else values.max() / 2
    for i in range(len(MODES)):
        for j in range(len(MODES)):
            label = f"{values[i, j]:.2f}" if normalized else f"{int(values[i, j])}"
            ax.text(j, i, label, ha="center", va="center",
                    color="white" if values[i, j] > threshold else "black", fontsize=12)
    ax.set_xticks(range(len(MODES)), labels=MODES)
    ax.set_yticks(range(len(MODES)), labels=MODES)
    ax.set_xlabel("Modo predicho")
    ax.set_ylabel("Modo real")
    ax.set_title(title, pad=14, fontweight="bold")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plot_dir = ROOT / "pipeline_v4/calibration_and_diagnostics/modal_classification/figures_local"
    plot_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(plot_dir / filename, dpi=220, bbox_inches="tight")
    plt.show()

plot_confusion(cm, "Absolute confusion matrix — hybrid modal classifier OOF", "hybrid_oof_confusion_matrix_absolute.png")
plot_confusion(cm_normalized, "Confusion matrix normalized by true class — hybrid modal classifier OOF", "hybrid_oof_confusion_matrix_normalized.png", normalized=True)

In [ ]:
results = df[['deg']].copy(); results['y'] = y; results['pred'] = oof
by_deg = results.groupby('deg').apply(lambda g: pd.Series({'balanced_accuracy': balanced_accuracy_score(g.y,g.pred), 'macro_f1': f1_score(g.y,g.pred,average='macro')}), include_groups=False)
display(by_deg.loc[['Raw','L1','L2','L3']])


## Inference, quality guardrail, and configuration selection

In [ ]:
hybrid = create_modal_evaluator('hybrid')
rf_rollback = create_modal_evaluator('random_forest')
bayes = create_modal_evaluator('bayes')
print(type(hybrid).__name__, type(rf_rollback).__name__, type(bayes).__name__)
frame = pd.DataFrame({'caid':['DEMO']*20, 'trip':[1]*20, 'Speed [km/h]':np.linspace(12,30,20), 'local_timestamp':pd.date_range('2026-07-15',periods=20,freq='30s'), 'highway':['primary']*20, 'distance_m':[100.0]*20, 'near_bus_route':[0]*20, 'near_subway_line':[0]*20})
hybrid.raw_counts['DEMO_1'] = 20
hypotheses = {'Carro': frame}
print('Inferencia:', hybrid.select_final_mode(hypotheses)[:3])
short = {k:v.iloc[:10].copy() for k,v in hypotheses.items()}
print('Guardrail:', hybrid.select_final_mode(short)[0])  # Calidad insuficiente


## Supported serving backends

The hierarchical hybrid classifier is the production default. The previous three-level Random Forest model and the Bayesian classifier remain available as explicit alternatives.